# 06 — TensorFlow on CPU, then TPUStrategy TPU-ready

**Learning goal.** Train a tiny Keras MLP on synthetic data on CPU, then see the minimal change to use **TPUStrategy** on a Cloud TPU VM.

**What you'll observe.**
- A `keras.Sequential` MLP `.fit()` on synthetic data on CPU (a few epochs only).
- A guarded "TPU-ready" cell that resolves the TPU cluster, creates a `TPUStrategy`, and would rebuild the model under the strategy scope.

**Knobs to tune.**
- `BATCH`, `HIDDEN`, `EPOCHS`, `N_SAMPLES`.
- On a TPU VM, `TPU_NAME` (your TPU node's name) is used to resolve the cluster.

**Failure modes to watch.**
- TensorFlow may be missing — the cells are guarded.
- `TPUStrategy` needs the cluster resolver; outside a TPU VM it will not be able to connect.
- TPUStrategy's `.fit()` requires `tf.data` pipelines, not raw NumPy, on real TPUs.

In [ ]:
import sys, os
from pathlib import Path
# Add the parent of cloud_tpu_lab to sys.path so `cloud_tpu_lab.src.*` imports work.
HERE = Path(os.getcwd()).resolve()
_root = HERE
for _ in range(5):
    if (_root / "cloud_tpu_lab").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))
sys.path.insert(0, "..")
PLOT_DIR = Path("../artifacts/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", _root)
print("plot dir:", PLOT_DIR.resolve())

## CPU TensorFlow / Keras MLP

In [ ]:
try:
    import tensorflow as tf
    HAVE_TF = True
except Exception as e:
    HAVE_TF = False
    print("TensorFlow is not installed in this kernel:", e)
    print("Skipping TF cells.")

In [ ]:
if HAVE_TF:
    import numpy as np
    print("tf version:", tf.__version__)
    print("devices:", tf.config.list_physical_devices())

    BATCH, IN_DIM, HIDDEN, OUT, N_SAMPLES, EPOCHS = 32, 64, 64, 10, 1024, 2

    rng = np.random.default_rng(0)
    X = rng.standard_normal((N_SAMPLES, IN_DIM)).astype("float32")
    y = rng.integers(0, OUT, size=(N_SAMPLES,)).astype("int32")

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IN_DIM,)),
        tf.keras.layers.Dense(HIDDEN, activation="relu"),
        tf.keras.layers.Dense(HIDDEN, activation="relu"),
        tf.keras.layers.Dense(OUT),
    ])
    model.compile(optimizer="adam",
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=["accuracy"])
    history = model.fit(X, y, batch_size=BATCH, epochs=EPOCHS, verbose=2)
    print("final loss:", history.history["loss"][-1])
else:
    print("(skipped — no TF)")

## TPU-ready (only runs on Cloud TPU)

On a Cloud TPU VM you'd:

1. Resolve the cluster: `tf.distribute.cluster_resolver.TPUClusterResolver(tpu=TPU_NAME)`.
2. Connect: `tf.config.experimental_connect_to_cluster(resolver)`.
3. Initialize: `tf.tpu.experimental.initialize_tpu_system(resolver)`.
4. Wrap with `strategy = tf.distribute.TPUStrategy(resolver)` and build the model **inside** `with strategy.scope():`.

The cell below detects whether you're on a TPU VM. Off-TPU, it just prints what it would do.

In [ ]:
# TPU-ready (only runs on Cloud TPU; safely skips elsewhere)
if HAVE_TF:
    on_tpu = False
    try:
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu=os.environ.get("TPU_NAME", ""))
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)
        strategy = tf.distribute.TPUStrategy(resolver)
        on_tpu = True
        print("TPU strategy ready with", strategy.num_replicas_in_sync, "replicas")
    except Exception as e:
        print("TPU not available in this kernel:", repr(e))
        print("On a real Cloud TPU VM you would now build the model under `with strategy.scope():`")
        print("Example sketch (not executed here):")
        print("    with strategy.scope():")
        print("        model = build_keras_model()")
        print("        model.compile(...)")
        print("    model.fit(tf_dataset, epochs=EPOCHS)")
else:
    print("(skipped — no TF)")

## A note on data pipelines

On TPUs, **`tf.data` is the supported input path**. Raw NumPy arrays as in the CPU cell above can work for tiny demos, but real workloads should use:

```python
ds = tf.data.Dataset.from_tensor_slices((X, y))
ds = ds.shuffle(1024).batch(BATCH, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
```

The `drop_remainder=True` is important — `TPUStrategy` needs static batch shapes.

## Exercises

1. Add `tf.data.Dataset.from_tensor_slices(...)` with `prefetch(tf.data.AUTOTUNE)` on the CPU path. Does fit time change much for tiny `N_SAMPLES`?
2. Use `tf.profiler.experimental.start(...)` around `.fit()` and capture a profile. Inspect it (off-TPU it still works locally).
3. Replace the MLP with a tiny `keras.Sequential` CNN on the same synthetic data shaped as images. Compare per-step time.
4. On a real TPU VM, run the TPU-ready cell and report `strategy.num_replicas_in_sync`. Does it equal the chip count you provisioned?
5. What single thing about `tf.data` is most important when targeting a TPU? (Hint: `drop_remainder`.)